# Сильный режим GNN — обучение лестницы L3→L6 на Colab A100

Считает 4 «сильных» шага лестницы перехода (PNA / Multi-PNA+EU, большие окрестности)
на полном IBM AML HI-Small и пушит результаты обратно в репозиторий. L0–L2 (слабый
режим) уже посчитаны — здесь только L3–L6.

**Перед запуском:**
1. `Runtime → Change runtime type → A100` (желательно **High-RAM**).
2. Данные на Google Drive: `HI-Small_Trans.csv` и `HI-Small_Patterns.txt` (не в git).
3. GitHub **Personal Access Token** (scope `repo` / contents:write) для push.

Запускай ячейки сверху вниз.

## 1. Проверка GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "Нет GPU! Runtime → Change runtime type → A100 (High-RAM)"
name = torch.cuda.get_device_name(0)
print("GPU:", name)
if "A100" not in name:
    print("\n⚠️  Не A100. Конфиги L3–L6 рассчитаны на A100 40GB ([50,50,50]/batch 4096).")
    print("    На L4/T4 снизь num_neighbors→[25,25,25] и batch_size→1024 ОДИНАКОВО по всем L3–L6,")
    print("    иначе прирост нельзя честно приписать рычагу.")

## 2. Клонирование репозитория

In [ ]:
import os
REPO = "/content/gnn-aml-graphs-kr"
if not os.path.exists(REPO):
    !git clone https://github.com/Ezzenin/gnn-aml-graphs-kr.git {REPO}
%cd {REPO}
!git pull --quiet
print("repo:", REPO)

## 3. Зависимости

In [ ]:
!pip -q install torch_geometric xgboost networkx pyyaml wandb 2>&1 | tail -2
import torch, torch_geometric
print("torch", torch.__version__, "| pyg", torch_geometric.__version__)

## 4. Данные с Google Drive
Поправь `DRIVE_DATA`, если файлы лежат в другой папке Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
DRIVE_DATA = "/content/drive/MyDrive/ibm_aml"   # ← путь к папке с HI-Small_*.csv/.txt на Drive
os.makedirs(f"{REPO}/data/ibm_aml", exist_ok=True)
for fn in ["HI-Small_Trans.csv", "HI-Small_Patterns.txt"]:
    src = os.path.join(DRIVE_DATA, fn)
    assert os.path.exists(src), f"Нет файла {src} — залей данные на Drive в {DRIVE_DATA}"
    shutil.copy(src, f"{REPO}/data/ibm_aml/{fn}")
print("Данные на месте:", os.listdir(f"{REPO}/data/ibm_aml"))

## 5. Git-личность + токен (для push)
Токен вводится скрыто и не сохраняется в выводе.

In [ ]:
import subprocess, getpass
token = getpass.getpass("GitHub PAT (scope repo / contents:write): ").strip()
subprocess.run(["git","config","user.name","Ezzenin"], cwd=REPO, check=True)
subprocess.run(["git","config","user.email","eseninaleksandr@gmail.com"], cwd=REPO, check=True)
subprocess.run(["git","remote","set-url","origin",
                f"https://{token}@github.com/Ezzenin/gnn-aml-graphs-kr.git"], cwd=REPO, check=True)
del token
print("git настроен (коммиты от Ezzenin, токен не в выводе)")

## 6. SMOKE — быстрый чек, что PNA-путь не падает
Подвыборка + урезанные окрестности, 2 эпохи. Это НЕ финальные числа — только проверка
VRAM/RAM/кода перед многочасовым полным прогоном.

In [ ]:
import yaml, sys, subprocess
base = yaml.safe_load(open(f"{REPO}/configs/ibm_pna_fulldata.yaml"))
base["dataset"]["max_rows"] = 500000
base["train"]["num_neighbors"] = [25, 25, 25]
base["train"]["batch_size"] = 1024
base["train"]["epochs"] = 2
base["train"]["patience"] = 2
base["output_name"] = "ibm_pna_smoke"
base["save_checkpoint"] = False
yaml.safe_dump(base, open(f"{REPO}/configs/_pna_smoke.yaml","w"), allow_unicode=True)
rc = subprocess.run([sys.executable,"-m","src.train_edge","--config","configs/_pna_smoke.yaml"], cwd=REPO).returncode
print("\nSMOKE rc =", rc, "(0 = ок, можно гнать полные прогоны ниже)")

## 7. Полные прогоны L3 → L6
~часы на A100. Если конфиг падает по OOM — снизь `batch_size`/`num_neighbors`
**одинаково во всех 4 конфигах** (`configs/ibm_*pna*_fulldata.yaml`,
`configs/ibm_multignn_big_fulldata.yaml`) и перезапусти ячейку.

In [ ]:
import sys, subprocess
LADDER = ["ibm_multignn_big_fulldata",  # L3
          "ibm_pna_fulldata",            # L4
          "ibm_multipna_fulldata",       # L5
          "ibm_multipna_eu_fulldata"]    # L6 (целевой)
for c in LADDER:
    print(f"\n========== {c} ==========")
    rc = subprocess.run([sys.executable,"-m","src.train_edge","--config",f"configs/{c}.yaml"], cwd=REPO).returncode
    if rc != 0:
        print(f"\n!! {c} упал (rc={rc}). Останавливаюсь.")
        print("   OOM → снизь batch_size/num_neighbors ОДИНАКОВО по L3–L6 и перезапусти с этого конфига.")
        break
else:
    print("\n✅ Все 4 прогона завершены.")

## 8. (опционально) Репродукция под протоколом Egressy 60/20/20
Для внешней сопоставимости с литературой. Можно пропустить.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable,"-m","src.train_edge","--config","configs/ibm_multipna_eu_egressy.yaml"], cwd=REPO)

## 9. Пересборка лестницы (таблица + фигура)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable,"-m","src.compare","--ibm"], cwd=REPO)
print("\n--- ibm_ladder.md ---")
print(open(f"{REPO}/results/ibm_ladder.md").read()[:1200])

## 10. Коммит + push результатов в репозиторий
Пушатся только `results/*.json` и `*.png`/`*.md` (чекпоинты `.pt` и данные — в .gitignore).
Smoke-артефакты удаляются перед коммитом.

In [ ]:
import os, subprocess
subprocess.run("rm -f configs/_pna_smoke.yaml results/ibm_pna_smoke*", cwd=REPO, shell=True)
files = [
  "results/ibm_multignn_big_fulldata_metrics.json", "results/ibm_multignn_big_fulldata_pr_curve.png",
  "results/ibm_pna_fulldata_metrics.json",          "results/ibm_pna_fulldata_pr_curve.png",
  "results/ibm_multipna_fulldata_metrics.json",     "results/ibm_multipna_fulldata_pr_curve.png",
  "results/ibm_multipna_eu_fulldata_metrics.json",  "results/ibm_multipna_eu_fulldata_pr_curve.png",
  "results/ibm_multipna_eu_egressy_metrics.json",   "results/ibm_multipna_eu_egressy_pr_curve.png",
  "results/ibm_ladder.md", "results/transition_ladder.png",
  "results/per_pattern.md", "results/per_pattern.png", "results/ibm_comparison_fulldata.md",
]
files = [f for f in files if os.path.exists(os.path.join(REPO, f))]
subprocess.run(["git","add"] + files, cwd=REPO)
subprocess.run(["git","commit","-m","feat: сильный режим — прогон лестницы L3-L6 (Multi-PNA+EU) на A100"], cwd=REPO)
subprocess.run(["git","push","origin","main"], cwd=REPO)
print("\nГотово. На Mac: git pull && python -m src.compare --ibm")